# 하천 유량 예측 — 시계열 EDA와 모델 (Saugeen River)

- 데이터: 캐나다 소지강 일평균 유량 (23,741행 · 65년)
- 목표: 유량(㎥/s) 예측 — 단일 시계열
- 흐름: 불러오기 → 시계열 EDA → 형식 변환 → 학습 → 해석
- 참고: 데이터 소개 data_saugeen_river.txt

- 이 데이터의 특징
  · **가장 깨끗** (결측 0 · 빈 날짜 0) → 전처리보다 모델·해석에 집중
  · **치우친 분포** (최대가 중앙값의 37배) → 로그 변환
  · 단일 시계열 → 시계열의 기본에 집중

## 1. 불러오기

- 이미 3열 형식 · 결측 없음
- 시계열이 1개뿐 (item_id = "T1")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from autogluon.common.utils.resource_utils import ResourceManager
ResourceManager.get_cpu_count = staticmethod(lambda **kwargs: 16)

df = pd.read_csv("saugeen_river_flow.csv", parse_dates=["timestamp"])
print(df.shape)                       # (23741, 3)
print("시계열 개수:", df["item_id"].nunique())
print("기간:", df["timestamp"].min().date(), "~", df["timestamp"].max().date())
df.head()

## 2. 시계열 EDA

### 2-1. target 분포 — 극단적 치우침 (핵심)

- 평소에는 낮고, 홍수 때만 급증
- 최대(640)가 중앙값(17.3)의 약 37배

In [ ]:
print(df["target"].describe().round(2))

fig, ax = plt.subplots(1, 2, figsize=(11,3))
df["target"].hist(bins=60, ax=ax[0]); ax[0].set_title("flow (raw)")
np.log(df["target"]).hist(bins=60, ax=ax[1])
ax[1].set_title("log(flow)")
plt.show()
# 원본은 왼쪽에 몰림 → 로그 변환하면 완만해짐

### 2-2. 계절성 — 융설과 갈수

- 4월 최다(눈 녹아 흐름), 8월 최소(여름 갈수)
- 연 주기가 뚜렷

In [ ]:
(df.assign(m=df["timestamp"].dt.month)
   .groupby("m")["target"].mean()
   .plot(kind="bar", figsize=(8,3), title="mean flow by month"))
plt.show()
# 4월 피크(융설) · 8월 최저(갈수)

### 2-3. 65년 장기 추세

- 오랜 기간이라 장기 변화를 볼 수 있음

In [ ]:
(df.assign(y=df["timestamp"].dt.year)
   .groupby("y")["target"].mean()
   .plot(figsize=(11,3), title="yearly mean flow (1915-1979)"))
plt.show()
# 뚜렷한 추세가 있는가? 원인은 무엇일까?

### 2-4. 계절 분해 (강의 49p)

- 원본을 추세 + 계절성 + 잔차로 분리
- period=365 (연 주기) — 최근 10년만

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

s = df.set_index("timestamp")["target"].iloc[-3650:]   # 최근 10년
result = seasonal_decompose(s, model="additive", period=365)
result.plot()
plt.gcf().set_size_inches(11, 7)
plt.show()

## 3. 시계열 형식 변환

- 결측·빈 날짜가 없어 변환이 단순
- convert_frequency 후 행수가 그대로면 정상

In [ ]:
from autogluon.timeseries import TimeSeriesDataFrame

ts = TimeSeriesDataFrame.from_data_frame(
    df, id_column="item_id", timestamp_column="timestamp")

before = len(ts)
ts = ts.convert_frequency(freq="D")
print(f"{before} → {len(ts)}")   # 같으면 빠진 날짜 없음

## 4. 학습

- prediction_length = 14 (2주 앞)
- 치우친 분포이므로 평가 지표 선택이 중요

In [ ]:
from autogluon.timeseries import TimeSeriesPredictor

prediction_length = 14
train_data, test_data = ts.train_test_split(prediction_length)

predictor = TimeSeriesPredictor(
    prediction_length=prediction_length,
    target="target",
    freq="D",
    eval_metric="RMSE",     # 큰 오차(홍수 실패)에 민감
).fit(train_data, presets="medium_quality", time_limit=600)

In [ ]:
predictor.leaderboard(test_data)
# SeasonalNaive 순위 확인 — 계절성 뚜렷하면 상위에 옴

## 5. 해석

In [ ]:
predictions = predictor.predict(train_data)

predictor.plot(
    data=test_data,
    predictions=predictions,
    quantile_levels=[0.1, 0.9],
    max_history_length=180,
)
plt.show()

### 5-1. 확인할 점 (강의자료 70~71p)

- 예측선이 계절 패턴을 따라가는가
- 실제값이 예측 구간(P10~P90) 안에 들어오는가
- 크게 벗어난 날이 있는가 → 홍수일 가능성

- **개별 홍수는 예측할 수 없다**
  · 이 데이터에는 강수량 정보가 없음
  · 모델은 "봄에 유량이 오른다"는 계절 패턴만 학습
  · "이번 주에 폭우가 온다"는 알 수 없음
  → 강의 71p 폭설 사례와 같은 구조

### 5-2. 로그 변환 비교 (선택 실험)

- 치우친 분포에 로그 변환이 도움이 되는지 확인
- 예측값을 원래 단위로 되돌려(np.exp) 비교해야 함

In [ ]:
# (선택) 로그 타깃으로 학습해 비교
# df["log_target"] = np.log(df["target"])
# → 같은 절차로 학습 후, np.exp로 되돌려 RMSE 비교

## 정리

- 가장 깨끗한 데이터(결측·빈 날짜 0) → 전처리 부담 없음
- 극단적 치우침(최대 37배) → 로그 변환 검토
- 계절성 뚜렷(4월 융설 · 8월 갈수) → SeasonalNaive 확인
- 65년 장기 → 추세·계절성 분해 (49p)
- 개별 홍수는 예측 불가 (강수 데이터 없음)

- 데이터가 깨끗해도 예측이 쉬운 것은 아니다
  → 필요한 정보(강수)가 없으면 한계가 있다